# Frozen Perch v2 + MLP

Run data preparation first. Local: use the project kernel after
`uv sync --extra notebooks --extra deep-learning --extra audio --extra perch`.
Kaggle/Colab: install a freshly built project bundle first (see `docs/remote.md`).
This notebook uses the same CLI, folds, sampler and reports as EfficientNet.

The first run downloads 45 MB of pinned ONNX weights and caches embeddings.
Subsequent runs train only the head. For offline Kaggle, attach the weights and
a complete cache as a Dataset and edit `ENCODER_PATH` and `CACHE_DIR` below.

In [1]:
import json

import torch

from birdclef.config import RuntimeConfig, TrainConfig, get_config
from birdclef.models import AudioExperimentConfig, AudioModelConfig, EncoderName
from birdclef.models.classifier.perch.download import download_perch
from birdclef.runtime import apply_environment_defaults

apply_environment_defaults()
defaults = get_config(verbose=False)

RUN_NAME = "perch/mlp"
SMOKE_RUN = False  # First check: 32 examples per source, one epoch, full-fold validation.
WANDB_OFFLINE = False
CACHE_DIR = defaults.data.cache_dir  # Kaggle Dataset: /kaggle/input/<dataset>/cache
ENCODER_PATH = CACHE_DIR / "models/perch_v2_backbone.onnx"
PREPARED_DIR = defaults.data.output_dir / "data_prep"  # Kaggle: bundle / "data_prep"

Detected environment: local
Loading default Config...


In [2]:
# !uv pip uninstall onnxruntime
# !uv pip install onnxruntime-gpu==1.23.2

In [3]:
# On Kaggle without internet, point ENCODER_PATH at an attached ONNX file instead.
download_perch(ENCODER_PATH)
torch.set_num_threads(4)
config = get_config(
    AudioExperimentConfig,
    fold=0,
    smoke=SMOKE_RUN,
    data=defaults.data.model_copy(update={"cache_dir": CACHE_DIR}),
    model=AudioModelConfig(
        encoder=EncoderName.PERCH,
        encoder_path=ENCODER_PATH,
        encoder_device="cuda",
        unfreeze_epoch=None,
        hidden_dim=512,
        dropout=0.3,
        head_lr=1e-3,
    ),
    tracking={"offline": WANDB_OFFLINE},
    runtime=RuntimeConfig(accelerator="gpu"),
    loader={"batch_size": 256, "val_batch_size": 256,
            "predict_batch_size": 16, "num_workers": 2, "seed": 42},
    train=TrainConfig(
        max_epochs=1 if SMOKE_RUN else 20,
        checkpoint_metric="val/auc", checkpoint_mode="max",
        early_stopping_patience=5, log_every_n_steps=1 if SMOKE_RUN else 10,
    ),
)
RUN_DIR = config.data.output_dir / config.run_name(RUN_NAME)
config_path = config.data.output_dir / "configs" / f"{config.run_name(RUN_NAME)}.json"
config_path.parent.mkdir(parents=True, exist_ok=True)
config_path.write_text(json.dumps(config.safe_dump(), indent=2), encoding="utf-8")
print(RUN_DIR)

                                        AudioExperimentConfig                                        
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Setting                       ┃ Value                                                             ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ app_name                      │ birdclef                                                          │
│ logging.directory             │ C:\_GitHub\birdclef-2026\logs                                     │
│ logging.level                 │ INFO                                                              │
│ logging.formatter             │ ColoredFormatter                                                  │
│ logging.date_format           │ %d-%m-%Y %H:%M:%S                                                 │
│ logging.full_color            │ True                                                              │
│ logging.include_function      │ True                                                              │
│ logging.success_level         │ 69                                                                │
├───────────────────────────────┼───────────────────────────────────────────────────────────────────┤
│ data.competition_dir          │ C:\_GitHub\birdclef-2026\data\competition                         │
│ data.cache_dir                │ C:\_GitHub\birdclef-2026\data\cache                               │
│ data.output_dir               │ C:\_GitHub\birdclef-2026\outputs                                  │
├───────────────────────────────┼───────────────────────────────────────────────────────────────────┤
│ split.validation_fraction     │ 0.2                                                               │
│ split.seed                    │ 42                                                                │
│ split.group_column            │ None                                                              │
│ split.stratify                │ False                                                             │
├───────────────────────────────┼───────────────────────────────────────────────────────────────────┤
│ tracking.project              │ birdclef-2026                                                     │
│ tracking.entity               │ dmykhailov-kyiv-school-of-economics                               │
│ tracking.offline              │ False                                                             │
│ tracking.api_key              │ **********                                                        │
├───────────────────────────────┼───────────────────────────────────────────────────────────────────┤
│ loader.batch_size             │ 256                                                               │
│ loader.val_batch_size         │ 256                                                               │
│ loader.predict_batch_size     │ 16                                                                │
│ loader.num_workers            │ 2                                                                 │
│ loader.pin_memory             │ False                                                             │
│ loader.seed                   │ 42                                                                │
│ loader.soundscape_probability │ 0.1                                                               │
├───────────────────────────────┼───────────────────────────────────────────────────────────────────┤
│ lr_scheduler.name             │ none                                                              │
│ lr_scheduler.t_max            │ 10                                                                │
│ lr_scheduler.min_lr           │ 0.0                                                               │
│ lr_scheduler.factor           │ 0.1                                                               │
│ lr_scheduler.patienc

C:\_GitHub\birdclef-2026\outputs\perch\mlp


## Train

Perch extraction uses `model.encoder_device` (`cpu` or `cuda`); the head uses
`runtime.accelerator`. CUDA extraction requires GPU ONNX Runtime. Focal recordings use
one centered crop, short clips are padded; soundscapes use exact annotated windows.
Full-fold validation can make the first smoke run take several minutes.

Use `SMOKE_RUN=False` for a full experiment. Keep a new run name for a changed head,
fold or source mix; the embedding cache is reused. Keep fold/seed/source mix equal
when comparing with EfficientNet, and account for the different focal crop policy.

In [4]:
%run -m birdclef.models.classifier.fit -- --config "{config_path}" --name "{RUN_NAME}" --prepared-dir "{PREPARED_DIR}"

W1029 05:14:27.867000 17772 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


29-10-2026 05:14:29  INFO      [models/classifier/fit:68]  Run: perch/mlp; training examples: 35769; validation windows: 149


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.forge.coreweave.com.
wandb: Appending key for api.forge.coreweave.com to your netrc file: C:\Users\dennm\_netrc
wandb: Currently logged in as: dmykhailov (dmykhailov-kyiv-school-of-economics) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


Seed set to 42


Output()

...

Output()

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Restoring states from the checkpoint path at C:\_GitHub\birdclef-2026\outputs\perch\mlp\checkpoints\epoch004.ckpt
c:\_GitHub\birdclef-2026\.venv\Lib\site-packages\lightning\pytorch\trainer\call.py:283: Be aware that when using `ckpt_path`, callbacks used to create the checkpoint need to be provided during `Trainer` instantiation. Please add the following callbacks: ["EarlyStopping{'monitor': 'val/auc', 'mode': 'max'}", "ModelCheckpoint{'monitor': 'val/auc', 'mode': 'max', 'every_n_train_steps': 0, 'every_n_epochs': 1, 'train_time_interval': None}", "ModelCheckpoint{'monitor': None, 'mode': 'min', 'every_n_train_steps': 0, 'every_n_epoc

Predicting: |          | 0/? [00:00<?, ?it/s]

best/val_auc,▁
epoch,▁▁▁▁▁▁▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▆▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇██
lr-AdamW,▁▁▁▁▁▁▁▁▁▁
perf/fit_seconds,▁
train/loss_epoch,█▂▂▂▂▁▁▁▁▁
train/loss_step,█▆▄▃▃▃▃▃▃▃▃▃▂▃▂▂▂▂▃▂▂▂▂▂▂▂▂▂▁▁▂▁▁▂▂▁▂▂▁▁
trainer/global_step,▁▁▁▁▁▂▂▂▂▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇█████
val/auc,▁▅▇▇█▅▇▇▇▇
val/auc_classes,▁▁▁▁▁▁▁▁▁▁
val/loss,▁▂▃▄▄▆▅▇▇█
best/val_auc,0.92631


Saved training run: C:\_GitHub\birdclef-2026\outputs\perch\mlp


In [5]:
import pandas as pd
from IPython.display import display

display(pd.read_json(RUN_DIR / "validation-metrics.json", typ="series"))
display(pd.read_csv(RUN_DIR / "validation-class-metrics.csv").sort_values("auc", ascending=False).head(15))

auc                    0.926314
auc_classes                  28
windows                     149
protocol       file-gkf5/fold-0
dtype: object

,species,positive,negative,auc
48,47158son19,1.0,148.0,1.000000
46,47158son17,9.0,140.0,0.996825
103,compau,4.0,145.0,0.996552
21,24321,41.0,108.0,0.994693
56,516975,5.0,144.0,0.993056
225,whtdov,10.0,139.0,0.992086
97,chacha1,16.0,133.0,0.988252
165,redjun,6.0,143.0,0.980186
60,555146,52.0,97.0,0.978390
15,23158,48.0,101.0,0.976073


## CPU inference check

This uses the encoder saved inside the run, bypassing the training cache. Time includes
model loading, audio decoding, Perch and the MLP. One minute is a sanity check;
measure the full test workload on Kaggle before judging submission runtime.

In [6]:
from time import perf_counter

from birdclef.competition import ROW_ID, soundscape_windows
from birdclef.submission import predict_runs

windows = soundscape_windows(config.data.competition_dir / "train_soundscapes")
minute = windows.loc[windows.filename.eq(windows.filename.iloc[0])].reset_index(drop=True)
classes = json.loads((RUN_DIR / "classes.json").read_text(encoding="utf-8"))
template = minute[[ROW_ID]].reindex(columns=[ROW_ID, *classes], fill_value=0.0)
started = perf_counter()
predictions = predict_runs([RUN_DIR], minute, template, batch_size=16)
benchmark = {"seconds": perf_counter() - started, "windows": len(minute),
             "includes": "model load, decode, Perch CPU, MLP", "onnx_threads": 4}
(RUN_DIR / "cpu-benchmark.json").write_text(json.dumps(benchmark, indent=2), encoding="utf-8")
display(pd.Series(benchmark))
display(predictions.head())

Model: C:\_GitHub\birdclef-2026\outputs\perch\mlp


Output()

GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
c:\_GitHub\birdclef-2026\.venv\Lib\site-packages\lightning\pytorch\trainer\setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Restoring states from the checkpoint path at C:\_GitHub\birdclef-2026\outputs\perch\mlp\checkpoints\epoch004.ckpt
c:\_GitHub\birdclef-2026\.venv\Lib\site-packages\lightning\pytorch\trainer\call.py:283: Be aware that when using `ckpt_path`, callbacks used to create the checkpoint need to be provided during `Trainer` instantiation. Please add the following callbacks: ["EarlyStopping{'monitor': 'val/auc', 'mode': 'max'}", "ModelCheckpoint{'monitor': 'val/auc', 'mode': 'max', 'every_n_train_steps': 0, 'every_n_epochs

Predicting: |          | 0/? [00:00<?, ?it/s]

seconds                                   1.635218
windows                                         12
includes        model load, decode, Perch CPU, MLP
onnx_threads                                     4
dtype: object

,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Train_0001_S08_20250606_030007_5,0.000137,0.020741,0.000028,0.000012,0.000149,0.000035,0.000536,0.000126,0.000060,...,0.000028,0.000682,0.000007,0.000040,0.000021,0.000087,0.000019,0.000003,0.000100,0.000008
1,BC2026_Train_0001_S08_20250606_030007_10,0.000244,0.060229,0.000041,0.000007,0.000197,0.000054,0.000849,0.000209,0.000055,...,0.000029,0.000348,0.000006,0.000048,0.000012,0.000025,0.000034,0.000001,0.000133,0.000006
2,BC2026_Train_0001_S08_20250606_030007_15,0.000231,0.037080,0.000023,0.000018,0.000197,0.000028,0.000836,0.000140,0.000039,...,0.000016,0.000490,0.000004,0.000028,0.000016,0.000025,0.000027,0.000001,0.000095,0.000011
3,BC2026_Train_0001_S08_20250606_030007_20,0.000423,0.061692,0.000056,0.000010,0.000402,0.000085,0.000516,0.000190,0.000149,...,0.000029,0.000705,0.000006,0.000084,0.000023,0.000061,0.000046,0.000002,0.000102,0.000005
4,BC2026_Train_0001_S08_20250606_030007_25,0.000358,0.071136,0.000045,0.000013,0.000294,0.000055,0.001326,0.000286,0.000119,...,0.000046,0.000540,0.000006,0.000043,0.000019,0.000047,0.000047,0.000001,0.000173,0.000011


## Export

Run `python -m birdclef.submission.export --run <RUN_DIR> --all` to package the
checkpoint and Perch weights. The existing Kaggle inference notebook accepts both
EfficientNet and Perch runs, including ensembles. Rebuild the wheelhouse to include
ONNX Runtime. Upload the generated ZIP as a Kaggle Dataset.